In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Chargement
df = pd.read_csv("data/processed/features_complet_16_26_clean.csv")
df['_date'] = pd.to_datetime(df['_date'])

# Création de la cible binaire y (1 = Victoire Domicile, 0 = Nul/Défaite)
if 'target_win' not in df.columns:
    df['target_win'] = (df['home_goals'] > df['away_goals']).astype(int)

# Scission Passé vs Futur
df_train_val = df[df['_date'] < '2026-01-01'].copy()
df_predict_2026 = df[df['_date'] >= '2026-01-01'].copy()

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import torch

# [Ce bloc remplace l'ancienne séparation/normalisation]

# 1. Isolation des features et de la cible
colonnes_a_exclure = ['_date', '_home_team', '_away_team', '_tournament', 'home_goals', 'away_goals', 'target_win']
features_cols = [c for c in df_train_val.columns if c not in colonnes_a_exclure]

X = df_train_val[features_cols].values
y = df_train_val['target_win'].values
X_2026 = df_predict_2026[features_cols].values

# Séparation temporelle Train/Val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# 2. Colmatage des trous mathématiques (Imputation par la moyenne)
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_2026_imputed = imputer.transform(X_2026)

# 3. Normalisation stricte
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_2026_scaled = scaler.transform(X_2026_imputed)

# 4. Conversion en Tenseurs PyTorch
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

X_2026_tensor = torch.tensor(X_2026_scaled, dtype=torch.float32)

print("Imputation et conversion réussies. Plus aucun NaN dans les tenseurs.")

print(f"Nombre de variables en entrée du réseau : {len(features_cols)}")
print(f"Tenseur d'entraînement : {X_train_tensor.shape}")
print(f"Tenseur de validation  : {X_val_tensor.shape}")

Imputation et conversion réussies. Plus aucun NaN dans les tenseurs.
Nombre de variables en entrée du réseau : 29
Tenseur d'entraînement : torch.Size([6362, 29])
Tenseur de validation  : torch.Size([1591, 29])


In [30]:
import torch.nn as nn
import torch.optim as optim

class MatchPredictor(nn.Module):
    def __init__(self, input_dim):
        super(MatchPredictor, self).__init__()
        
        # Couche cachée 1
        self.layer1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2) # Sécurité contre le surapprentissage
        
        # Couche cachée 2
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        
        # Couche de sortie
        self.output_layer = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid() # Transformation en probabilité (0 à 1)

    def forward(self, x):
        # Passage des données à travers le réseau
        x = self.dropout1(self.relu1(self.layer1(x)))
        x = self.dropout2(self.relu2(self.layer2(x)))
        x = self.sigmoid(self.output_layer(x))
        return x

# 1. Instanciation dynamique du modèle
input_dim = X_train_tensor.shape[1]
model = MatchPredictor(input_dim)

# 2. Définition des règles mathématiques d'apprentissage
# BCELoss (Binary Cross Entropy) est la fonction standard pour comparer une probabilité à un résultat 1 ou 0
criterion = nn.BCELoss() 

# Adam est l'optimiseur qui va corriger les erreurs du modèle (Taux d'apprentissage de 0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Architecture du réseau instanciée :")
print(model)

Architecture du réseau instanciée :
MatchPredictor(
  (layer1): Linear(in_features=29, out_features=64, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.2, inplace=False)
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. Préparation des DataLoader pour le Mini-Batch Gradient Descent
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Taille de batch standard pour stabiliser les gradients
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 2. Configuration des paramètres d'exécution
epochs = 80
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"Début de l'entraînement sur {X_train_tensor.shape[0]} matchs...")
print(f"Validation sur {X_val_tensor.shape[0]} matchs...\n")

for epoch in range(epochs):
    model.train() # Mode entraînement : active le Dropout
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()          # Réinitialisation des gradients résiduels
        outputs = model(batch_X)       # Forward pass : calcul de la prédiction
        loss = criterion(outputs, batch_y) # Calcul de l'erreur (Binary Cross Entropy)
        loss.backward()                # Backward pass : calcul des gradients
        optimizer.step()               # Optimisation : mise à jour des poids
        
        train_loss += loss.item() * batch_X.size(0)
        predictions = (outputs >= 0.5).float()
        correct_train += (predictions == batch_y).sum().item()
        total_train += batch_y.size(0)
        
    # Évaluation systématique sur le set de validation (généralisation)
    model.eval() # Mode évaluation : désactive le Dropout pour figer le réseau
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad(): # Désactivation des gradients pour économiser la mémoire RAM/VRAM
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            predictions = (outputs >= 0.5).float()
            correct_val += (predictions == batch_y).sum().item()
            total_val += batch_y.size(0)
            
    # Calcul des métriques moyennes pour cette époque
    epoch_train_loss = train_loss / total_train
    epoch_val_loss = val_loss / total_val
    epoch_train_acc = correct_train / total_train
    epoch_val_acc = correct_val / total_val
    
    # Enregistrement pour l'analyse des courbes
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_acc'].append(epoch_val_acc)
    
    # Affichage du suivi toutes les 10 époques
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Époque {epoch+1:02d}/{epochs} -> "
              f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} || "
              f"Train Acc: {epoch_train_acc*100:.2f}% | Val Acc: {epoch_val_acc*100:.2f}%")

# 3. Sauvegarde des poids du modèle
torch.save(model.state_dict(), "model_match_predictor.pth")
print("\nEntraînement terminé. Le fichier de poids 'model_match_predictor.pth' a été créé.")

Début de l'entraînement sur 6362 matchs...
Validation sur 1591 matchs...

Époque 01/80 -> Train Loss: 0.6019 | Val Loss: 0.5562 || Train Acc: 68.22% | Val Acc: 72.66%
Époque 10/80 -> Train Loss: 0.5457 | Val Loss: 0.5424 || Train Acc: 72.82% | Val Acc: 72.85%
Époque 20/80 -> Train Loss: 0.5370 | Val Loss: 0.5490 || Train Acc: 73.11% | Val Acc: 73.10%
Époque 30/80 -> Train Loss: 0.5303 | Val Loss: 0.5525 || Train Acc: 73.61% | Val Acc: 73.35%
Époque 40/80 -> Train Loss: 0.5239 | Val Loss: 0.5507 || Train Acc: 74.16% | Val Acc: 73.35%
Époque 50/80 -> Train Loss: 0.5201 | Val Loss: 0.5586 || Train Acc: 74.24% | Val Acc: 73.10%
Époque 60/80 -> Train Loss: 0.5150 | Val Loss: 0.5606 || Train Acc: 75.12% | Val Acc: 72.91%
Époque 70/80 -> Train Loss: 0.5061 | Val Loss: 0.5696 || Train Acc: 75.42% | Val Acc: 73.54%
Époque 80/80 -> Train Loss: 0.5022 | Val Loss: 0.5681 || Train Acc: 75.35% | Val Acc: 72.85%

Entraînement國terminé. Le fichier de poids 'model_match_predictor.pth' a été créé.


In [32]:
import pandas as pd
import torch

# 1. Passage du modèle en mode évaluation (désactive le Dropout)
model.eval()

# 2. Calcul des prédictions (sans calcul de gradients pour aller vite)
with torch.no_grad():
    probabilities = model(X_2026_tensor).numpy().flatten()

# 3. Reconstruction du tableau de résultats pour l'utilisateur
df_predictions = df_predict_2026[['_date', '_home_team', '_away_team', '_tournament']].copy()
df_predictions['proba_victoire_home'] = probabilities
df_predictions['proba_nul_ou_defaite_home'] = 1 - probabilities

# 4. Attribution d'un résultat lisible (Seuil standard à 0.5)
df_predictions['prediction_ there'] = df_predictions['proba_victoire_home'].apply(
    lambda p: "Victoire Dom" if p > 0.55 else ("Défaite/Nul Dom" if p < 0.45 else "Match Serré / Nul")
)

# 5. Tri chronologique et sauvegarde
df_predictions = df_predictions.sort_values('_date').reset_index(drop=True)
chemin_predictions = "data/processed/predictions_coupe_du_monde_2026.csv"
df_predictions.to_csv(chemin_predictions, index=False)

print("Inférence terminée avec succès.")
print(f"Le fichier de prédictions a été sauvegardé ici : {chemin_predictions}\n")

# Affichage des 10 premières prédictions pour vérification immédiate
print(df_predictions[['_date', '_home_team', '_away_team', 'proba_victoire_home', 'prediction_ there']].head(10))

Inférence terminée avec succès.
Le fichier de prédictions a été sauvegardé ici : data/processed/predictions_coupe_du_monde_2026.csv

       _date    _home_team    _away_team  proba_victoire_home  \
0 2026-01-03       Senegal         Sudan             0.884584   
1 2026-01-03          Mali       Tunisia             0.769226   
2 2026-01-04       Morocco      Tanzania             0.901148   
3 2026-01-04  South Africa      Cameroon             0.508049   
4 2026-01-05       Nigeria    Mozambique             0.616073   
5 2026-01-05         Egypt         Benin             0.846248   
6 2026-01-06   Ivory Coast  Burkina Faso             0.710872   
7 2026-01-06       Algeria      DR Congo             0.799035   
8 2026-01-09          Mali       Senegal             0.630616   
9 2026-01-09       Morocco      Cameroon             0.627362   

   prediction_ there  
0       Victoire Dom  
1       Victoire Dom  
2       Victoire Dom  
3  Match Serré / Nul  
4       Victoire Dom  
5       Victo

In [ ]:
import pandas as pd
import numpy as np

# 1. Chargement des fichiers
df_preds = pd.read_csv("data/processed/predictions_coupe_du_monde_2026.csv")
df_features = pd.read_csv("data/raw/teams_match_features.csv", encoding='utf-8-sig')

# Calcul de la moyenne globale de buts du football international sur les 10 ans (constante de ligue)
moyenne_globale_buts = (df_features['home_goals'].mean() + df_features['away_goals'].mean()) / 2

def simuler_score_poisson_dynamique(row, moyenne_globale, n_simulations=10000):
    p_win_reseau = row['proba_victoire_home']
    
    # Récupération des capacités offensives et défensives basées sur la forme récente
    # Si les colonnes n'existent pas ou sont NaN, on utilise la moyenne globale (1.0)
    att_home = row.get('home_form_scored', moyenne_globale) / moyenne_globale
    def_away = row.get('away_form_conceded', moyenne_globale) / moyenne_globale
    att_away = row.get('away_form_scored', moyenne_globale) / moyenne_globale
    def_home = row.get('home_form_conceded', moyenne_globale) / moyenne_globale
    
    # Calcul des Lambdas initiaux spécifiques aux styles des deux équipes
    lambda_home = moyenne_globale * att_home * def_away
    lambda_away = moyenne_globale * att_away * def_home
    
    # AJUSTEMENT PAR LE RÉSEAU : 
    # On applique un multiplicateur basé sur l'écart entre la probabilité brute de Poisson 
    # et la certitude absolue de ton réseau de neurones PyTorch.
    ratio_force = p_win_reseau / (1.0 - p_win_reseau + 1e-5)
    
    # Rééquilibrage fin des lambdas pour respecter la prédiction maîtresse du réseau
    facteur_ajustement = np.sqrt(ratio_force / (lambda_home / (lambda_away + 1e-5) + 1e-5))
    lambda_home_ajuste = lambda_home * facteur_ajustement
    lambda_away_ajuste = lambda_away / (facteur_ajustement + 1e-5)
    
    # Sécurité anti-valeurs aberrantes
    lambda_home_ajuste = max(0.1, min(5.0, lambda_home_ajuste))
    lambda_away_ajuste = max(0.1, min(5.0, lambda_away_ajuste))

    # Tirages de Monte Carlo
    buts_home_simules = np.random.poisson(lambda_home_ajuste, n_simulations)
    buts_away_simules = np.random.poisson(lambda_away_ajuste, n_simulations)
    
    # Extraction de la combinaison de score la plus fréquente
    scores = list(zip(buts_home_simules, buts_away_simules))
    score_le_plus_probable = max(set(scores), key=scores.count)
    
    return score_le_plus_probable[0], score_le_plus_probable[1]

# 2. Génération des scores
scores_home = []
scores_away = []

print("Calcul des scores dynamiques (Poisson-Dixon-Coles + Réseau)...")
for idx, row in df_preds.iterrows():
    bh, ba = simuler_score_poisson_dynamique(row, moyenne_globale_buts)
    scores_home.append(bh)
    scores_away.append(ba)

df_preds['pred_home_goals'] = scores_home
df_preds['pred_away_goals'] = scores_away
df_preds['score_exact_predit'] = df_preds['pred_home_goals'].astype(str) + " - " + df_preds['pred_away_goals'].astype(str)

# Sauvegarde
df_preds.to_csv("data/processed/predictions_coupe_du_monde_2026_scores_dynamiques.csv", index=False)

# Affichage des résultats
print("\nTableau des scores réajustés par équipe :")
print(df_preds[['_date', '_home_team', '_away_team', 'proba_victoire_home', 'score_exact_predit']].head(15))

Calcul des scores dynamiques (Poisson-Dixon-Coles + Réseau)...

Tableau des scores réajustés par équipe :
         _date    _home_team    _away_team  proba_victoire_home  \
0   2026-01-03       Senegal         Sudan             0.884584   
1   2026-01-03          Mali       Tunisia             0.769226   
2   2026-01-04       Morocco      Tanzania             0.901148   
3   2026-01-04  South Africa      Cameroon             0.508049   
4   2026-01-05       Nigeria    Mozambique             0.616073   
5   2026-01-05         Egypt         Benin             0.846248   
6   2026-01-06   Ivory Coast  Burkina Faso             0.710872   
7   2026-01-06       Algeria      DR Congo             0.799035   
8   2026-01-09          Mali       Senegal             0.630616   
9   2026-01-09       Morocco      Cameroon             0.627362   
10  2026-01-10       Algeria       Nigeria             0.571283   
11  2026-01-10         Egypt   Ivory Coast             0.544645   
12  2026-01-14       Se

: 